In [5]:
import os
from openai import OpenAI

In [8]:
import os
from openai import OpenAI

client = OpenAI(
    # 若没有配置环境变量，请用阿里云百炼API Key将下行替换为：api_key="sk-xxx",
    # 各地域的API Key不同。获取API Key：https://help.aliyun.com/zh/model-studio/get-api-key
    api_key='sk-4803bfae5a4545eaadacc986b85c9919',
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen3.5-plus", # 此处以qwen3.5-plus为例，可按需更换模型名称。模型列表：https://help.aliyun.com/zh/model-studio/models
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://stimuli.oss-cn-beijing.aliyuncs.com/stimuli/0001.bmp"
                    },
                },
                {"type": "text", "text": "图中描绘的是什么景象?"},
            ],
        },
    ],
)
print(completion.choices[0].message.content)

这张图片描绘了一个**热闹的露天农贸市场**（或集市）的景象，充满了生活气息。以下是具体的细节描述：

1.  **丰富的农产品**：
    *   画面的视觉中心是一大堆堆积如山的**橙色胡萝卜**，看起来非常新鲜，数量很多。
    *   在胡萝卜的左侧和左下角，堆放着几个红色的网眼袋，里面装满了看起来像是**红洋葱**或者红土豆的农作物。
    *   这些蔬菜都铺在黑色的塑料布或防水布上。

2.  **忙碌的人群**：
    *   右侧有一位身穿**白色长袖衬衫**、浅色裤子并戴着**绿色帽子**的男子，他正弯着腰，似乎在整理货物、称重或者挑选商品。
    *   左侧有一位身穿**蓝色长袍**（可能是当地传统服饰）的人背对着镜头站立。
    *   背景中还有几位当地居民，有的戴着头巾，穿着色彩各异的衣服，正在摊位间忙碌或交谈。

3.  **环境细节**：
    *   图片的左上角可以看到一块**太阳能电池板**，这暗示了该地点可能位于电力供应不稳定的地区，或者是一个利用太阳能供电的偏远市场。
    *   天空湛蓝，光线明亮刺眼，表明这是一个晴朗的大白天。

总的来说，这是一幅展示当地居民日常买卖农产品场景的照片，很可能拍摄于非洲或某个发展中国家的乡村集市。


In [ ]:
import os
import pickle
import json
from glob import glob

from openai import OpenAI
from tqdm import tqdm

DASHSCOPE_API_KEY = "sk-4803bfae5a4545eaadacc986b85c9919"
IMAGE_BASE = "https://stimuli.oss-cn-beijing.aliyuncs.com/stimuli"
STIMULI_DIR = "/media/ubuntu/sda/TrippleN/stimuli"
CAPTION_PATH = "/media/ubuntu/sda/TrippleN/customize/coco_captions_1000x5.pkl"

keyword_list = [
    # "spatial layout",
    # "color attribute",
    # "action relation",
    "part-whole relation",
    "positional relation",
    "functional relation",
]

client = OpenAI(
    api_key=DASHSCOPE_API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

def load_paths_and_captions():
    files = sorted([f for f in os.listdir(STIMULI_DIR) if f.lower().endswith(".bmp")])
    paths = [os.path.join(STIMULI_DIR, f) for f in files]
    with open(CAPTION_PATH, "rb") as f:
        caps = pickle.load(f)
    return paths, caps

def build_prompt(caption, keyword):
    return (
        "Given the image and caption, first describe the background color style of the image with 3-5 words. "
        "Second, detect the TWO most important objects in the image. "
        f"Then, describe each of the objects and their relationship using: '{keyword}' with TWO sentences. "
        "For each sentence, use 5-10 words and as easy as possible.\n"
        "Then, detect the absolute position of the two objects in the image, and select from [right, left, top, bottom]. "
        "\"left\" and \"right\" should appear together for horizontal objects, and \"top\" and \"bottom\" should appear together for vertical objects. DO NOT mix.\n"
        "Example:\n"
        "### Background color style: Grayscale urban.\n"
        "### The Man [left]\n"
        "1. The man is standing near the sidewalk edge. The Man is close to the building wall.\n"
        "### The Suitcase [right]\n"
        "1. The suitcase is beside the man's foot. The Suitcase is placed on the street's curved edge.\n"
        f"Now, given the image I uploaded and the caption \"{caption}\", detect the two most important objects with absolute position, "
        f"describe them using '{keyword}' with EXACTLY the example format."
    )

def call_vlm(image_url, caption, keyword):
    prompt = build_prompt(caption, keyword)
    completion = client.chat.completions.create(
        model="qwen3.5-flash",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": image_url}},
                    {"type": "text", "text": prompt},
                ],
            }
        ],
    )
    return completion.choices[0].message.content

def main():
    paths, caps = load_paths_and_captions()
    paths, caps = paths[:1000], caps[:1000]
    os.makedirs("local_cache", exist_ok=True)
    for keyword in keyword_list:
        safe = keyword.replace(" ", "_")
        out_path = os.path.join("local_cache", f"descriptions_{safe}.jsonl")
        with open(out_path, "w", encoding="utf-8") as f:
            for idx, p in tqdm(enumerate(paths), total=len(paths), desc=keyword):
                basename = os.path.basename(p)
                image_url = IMAGE_BASE.rstrip("/") + "/" + basename
                c_arr = caps[idx]
                if hasattr(c_arr, "flat"):
                    if c_arr.size == 0:
                        caption = ""
                    else:
                        v = c_arr.flat[0]
                        caption = v.decode("utf-8", errors="ignore") if isinstance(v, bytes) else str(v)
                else:
                    caption = str(c_arr[0]) if c_arr else ""
                desc = call_vlm(image_url, caption, keyword)
                rec = {"index": idx, "keyword": keyword, "description": desc}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")

if __name__ == "__main__":
    main()

functional relation:   5%|▌         | 50/1000 [20:55<5:29:26, 20.81s/it] 